In [1]:
# Cài đặt PySpark
%pip install pyspark

In [2]:
# Import các thư viện cần thiết
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
import os

# Khởi tạo Spark Context
conf = SparkConf().setAppName("MovieRatingsAnalysis").setMaster("local[*]")
sc = SparkContext.getOrCreate(conf=conf)
spark = SparkSession.builder.appName("MovieRatingsAnalysis").getOrCreate()

print("Spark Context đã được khởi tạo thành công!")

Spark Context đã được khởi tạo thành công!


In [3]:
# Đọc dữ liệu từ các file
import os

# Check if running in Google Colab
if 'COLAB_GPU' in os.environ or 'KAGGLE_KERNEL_RUN_TYPE' in os.environ:
    data_path = "/content/"
else:
    data_path = "data/" # For local environment

# Đọc file movies.txt
movies_rdd = sc.textFile(data_path + "movies.txt")
print(f"Số lượng phim: {movies_rdd.count()}")

# Đọc file ratings_1.txt và ratings_2.txt
ratings_1_rdd = sc.textFile(data_path + "ratings_1.txt")
ratings_2_rdd = sc.textFile(data_path + "ratings_2.txt")

# Đọc file users.txt
users_rdd = sc.textFile(data_path + "users.txt")

print(f"Số lượng rating từ file 1: {ratings_1_rdd.count()}")
print(f"Số lượng rating từ file 2: {ratings_2_rdd.count()}")

# Hiển thị một số dòng dữ liệu mẫu
print("\nDữ liệu movies.txt (5 dòng đầu):")
for line in movies_rdd.take(5):
    print(line)

print("\nDữ liệu ratings_1.txt (5 dòng đầu):")
for line in ratings_1_rdd.take(5):
    print(line)

print("\nDữ liệu users.txt (5 dòng đầu):")
for line in users_rdd.take(5):
    print(line)

Số lượng phim: 50
Số lượng rating từ file 1: 84
Số lượng rating từ file 2: 100

Dữ liệu movies.txt (5 dòng đầu):
1001,The Godfather (1972),Crime|Drama
1002,The Shawshank Redemption (1994),Drama
1003,Schindler's List (1993),Biography|Drama|History
1004,Raging Bull (1980),Biography|Drama|Sport
1005,Casablanca (1942),Drama|Romance|War

Dữ liệu ratings_1.txt (5 dòng đầu):
7,1020,4.5,1577836800
23,1015,3.5,1577923200
45,1030,4.0,1578009600
12,1047,3.0,1578096000
38,1012,4.5,1578182400

Dữ liệu users.txt (5 dòng đầu):
1,M,28,3,12345
2,F,35,7,23456
3,M,42,2,34567
4,F,19,10,45678
5,M,31,1,56789


In [4]:
# Xử lý dữ liệu users
# Parse users.txt: UserID, Gender, Age, Occupation, Zip-code
def parse_user_age(line):
    parts = line.split(',')
    user_id = int(parts[0])
    gender = parts[1]  # M hoặc F
    age = int(parts[2])
    occupation = int(parts[3])
    zipcode = parts[4]
    return (user_id, age)

# Hàm phân nhóm tuổi
def get_age_group(age):
    if age <= 18:
        return "0-18"
    elif age <= 35:
        return "18-35"
    elif age <= 50:
        return "35-50"
    else:
        return "50+"

users_parsed = users_rdd.map(parse_user_age)
print("Users parsed with age (5 records):")
for user in users_parsed.take(5):
    user_id, age = user
    age_group = get_age_group(age)
    print(f"UserID: {user_id}, Age: {age}, Age Group: {age_group}")

# Tạo dictionary để tra cứu tuổi theo UserID
users_age_dict = users_parsed.collectAsMap()
# Tạo dictionary để tra cứu nhóm tuổi theo UserID
users_age_group_dict = {user_id: get_age_group(age) for user_id, age in users_age_dict.items()}

print(f"\nTổng số user trong dictionary: {len(users_age_dict)}")
print(f"Phân bố nhóm tuổi:")
age_groups = list(users_age_group_dict.values())
for group in ["0-18", "18-35", "35-50", "50+"]:
    count = age_groups.count(group)
    print(f"  {group}: {count} users")

Users parsed with age (5 records):
UserID: 1, Age: 28, Age Group: 18-35
UserID: 2, Age: 35, Age Group: 18-35
UserID: 3, Age: 42, Age Group: 35-50
UserID: 4, Age: 19, Age Group: 18-35
UserID: 5, Age: 31, Age Group: 18-35

Tổng số user trong dictionary: 50
Phân bố nhóm tuổi:
  0-18: 0 users
  18-35: 25 users
  35-50: 23 users
  50+: 2 users


In [5]:
# Xử lý dữ liệu movies
# Parse movies.txt: MovieID, Title, Genres
def parse_movie(line):
    parts = line.split(',', 2)  # Tách thành 3 phần: ID, Title, Genres
    movie_id = int(parts[0])
    title = parts[1]
    genres = parts[2] if len(parts) > 2 else ""
    return (movie_id, title)

movies_parsed = movies_rdd.map(parse_movie)
print("Movies parsed (5 records):")
for movie in movies_parsed.take(5):
    print(f"MovieID: {movie[0]}, Title: {movie[1]}")

# Tạo dictionary để tra cứu tên phim theo ID
movies_dict = movies_parsed.collectAsMap()
print(f"\nTổng số phim trong dictionary: {len(movies_dict)}")

Movies parsed (5 records):
MovieID: 1001, Title: The Godfather (1972)
MovieID: 1002, Title: The Shawshank Redemption (1994)
MovieID: 1003, Title: Schindler's List (1993)
MovieID: 1004, Title: Raging Bull (1980)
MovieID: 1005, Title: Casablanca (1942)

Tổng số phim trong dictionary: 50


In [6]:
# Xử lý dữ liệu ratings
# Parse ratings: UserID, MovieID, Rating, Timestamp
def parse_rating_with_user(line):
    parts = line.split(',')
    user_id = int(parts[0])
    movie_id = int(parts[1])
    rating = float(parts[2])
    timestamp = int(parts[3])
    return (user_id, movie_id, rating)

# Parse cả 2 file ratings
ratings_1_parsed = ratings_1_rdd.map(parse_rating_with_user)
ratings_2_parsed = ratings_2_rdd.map(parse_rating_with_user)

print("Ratings 1 parsed (5 records):")
for rating in ratings_1_parsed.take(5):
    print(f"UserID: {rating[0]}, MovieID: {rating[1]}, Rating: {rating[2]}")

print("\nRatings 2 parsed (5 records):")
for rating in ratings_2_parsed.take(5):
    print(f"UserID: {rating[0]}, MovieID: {rating[1]}, Rating: {rating[2]}")

# Gộp 2 RDD ratings lại
all_ratings = ratings_1_parsed.union(ratings_2_parsed)
print(f"\nTổng số ratings từ cả 2 file: {all_ratings.count()}")

Ratings 1 parsed (5 records):
UserID: 7, MovieID: 1020, Rating: 4.5
UserID: 23, MovieID: 1015, Rating: 3.5
UserID: 45, MovieID: 1030, Rating: 4.0
UserID: 12, MovieID: 1047, Rating: 3.0
UserID: 38, MovieID: 1012, Rating: 4.5

Ratings 2 parsed (5 records):
UserID: 12, MovieID: 1012, Rating: 3.5
UserID: 34, MovieID: 1039, Rating: 4.0
UserID: 27, MovieID: 1043, Rating: 4.5
UserID: 8, MovieID: 1020, Rating: 3.0
UserID: 19, MovieID: 1050, Rating: 4.0

Tổng số ratings từ cả 2 file: 184


In [7]:
# Thêm thông tin nhóm tuổi vào ratings
# all_ratings: (user_id, movie_id, rating)
# Thêm nhóm tuổi: (movie_id, (rating, age_group))
def add_age_group_to_rating(record):
    user_id, movie_id, rating = record
    age_group = users_age_group_dict.get(user_id, "Unknown")
    return (movie_id, (rating, age_group))

ratings_with_age_group = all_ratings.map(add_age_group_to_rating)

print("Ratings with age group (5 records):")
for record in ratings_with_age_group.take(5):
    print(f"MovieID: {record[0]}, Rating: {record[1][0]}, Age Group: {record[1][1]}")

# Lọc chỉ những rating có nhóm tuổi xác định
valid_ratings = ratings_with_age_group.filter(lambda x: x[1][1] != "Unknown")

print(f"\nTổng số ratings có nhóm tuổi hợp lệ: {valid_ratings.count()}")

# Tách ratings theo nhóm tuổi
age_groups = ["0-18", "18-35", "35-50", "50+"]
ratings_by_age = {}

for age_group in age_groups:
    ratings_by_age[age_group] = valid_ratings.filter(lambda x: x[1][1] == age_group).map(lambda x: (x[0], x[1][0]))  # (movie_id, rating)
    print(f"Số ratings từ nhóm {age_group}: {ratings_by_age[age_group].count()}")

Ratings with age group (5 records):
MovieID: 1020, Rating: 4.5, Age Group: 35-50
MovieID: 1015, Rating: 3.5, Age Group: 18-35
MovieID: 1030, Rating: 4.0, Age Group: 35-50
MovieID: 1047, Rating: 3.0, Age Group: 35-50
MovieID: 1012, Rating: 4.5, Age Group: 18-35

Tổng số ratings có nhóm tuổi hợp lệ: 184
Số ratings từ nhóm 0-18: 0
Số ratings từ nhóm 18-35: 78
Số ratings từ nhóm 35-50: 96
Số ratings từ nhóm 50+: 10


In [8]:
# Tính điểm trung bình cho từng nhóm tuổi
age_averages = {}

# Tính stats cho từng nhóm tuổi
for age_group in age_groups:
    age_stats = ratings_by_age[age_group].map(lambda x: (x[0], (x[1], 1))).reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1]))
    age_averages[age_group] = age_stats.map(lambda x: (x[0], x[1][0] / x[1][1]))  # (movie_id, average_rating)

# Tìm tất cả các movie_id có rating
all_movie_ids = valid_ratings.map(lambda x: x[0]).distinct()

# Tính kết quả cho tất cả phim
all_movie_ids_list = all_movie_ids.collect()
final_results = []

for movie_id in all_movie_ids_list:
    movie_title = movies_dict.get(movie_id, f"Unknown Movie {movie_id}")
    age_ratings = {}

    for age_group in age_groups:
        # Tìm rating cho movie_id này từ nhóm tuổi này
        movie_ratings = age_averages[age_group].filter(lambda x: x[0] == movie_id).collect()
        if movie_ratings:
            age_ratings[age_group] = movie_ratings[0][1]  # average_rating
        else:
            age_ratings[age_group] = None

    final_results.append((movie_id, movie_title, age_ratings))

# Hiển thị kết quả theo định dạng yêu cầu
for movie_id, title, age_ratings in final_results:
    ratings_str = ", ".join([
        f"{age_group}: {age_ratings[age_group]:.2f}" if age_ratings[age_group] is not None else f"{age_group}: NA"
        for age_group in age_groups
    ])
    print(f"{title} - [{ratings_str}]")

E.T. the Extra-Terrestrial (1982) - [0-18: NA, 18-35: 3.56, 35-50: 3.83, 50+: 3.00]
Psycho (1960) - [0-18: NA, 18-35: 4.50, 35-50: 3.50, 50+: NA]
Gladiator (2000) - [0-18: NA, 18-35: 3.44, 35-50: 3.81, 50+: 3.50]
Fight Club (1999) - [0-18: NA, 18-35: 3.50, 35-50: 3.50, 50+: 3.50]
The Lord of the Rings: The Fellowship of the Ring (2001) - [0-18: NA, 18-35: 4.00, 35-50: 3.83, 50+: NA]
The Terminator (1984) - [0-18: NA, 18-35: 4.17, 35-50: 4.05, 50+: 3.75]
The Godfather: Part II (1974) - [0-18: NA, 18-35: 3.78, 35-50: 4.25, 50+: NA]
The Silence of the Lambs (1991) - [0-18: NA, 18-35: 3.00, 35-50: 3.25, 50+: NA]
Mad Max: Fury Road (2015) - [0-18: NA, 18-35: 3.36, 35-50: 3.64, 50+: NA]
Lawrence of Arabia (1962) - [0-18: NA, 18-35: 3.60, 35-50: 3.29, 50+: 4.50]
Sunset Boulevard (1950) - [0-18: NA, 18-35: 4.17, 35-50: 4.50, 50+: NA]
The Social Network (2010) - [0-18: NA, 18-35: 4.00, 35-50: 3.67, 50+: NA]
No Country for Old Men (2007) - [0-18: NA, 18-35: 3.81, 35-50: 3.94, 50+: 4.00]
The Lord

In [9]:
# Dọn dẹp tài nguyên
sc.stop()
spark.stop()
print("Đã dừng Spark Context và Spark Session.")

Đã dừng Spark Context và Spark Session.
